# Entropy-Regularized SPMD

## Problems

Both problems are the team's environments (`grid_world_env.py`, `blackjack_env.py`), imported
here, not copied, so this notebook always uses the same rules as the baseline. Both are written with
**costs** (lower is better) and discount factor γ = 0.9.

Each environment provides `n_states`, `n_actions`, `gamma` and `step_one(state, action, rng)` returning
`(next_state, cost, done)`, which is everything the team's online estimator needs to sample Q.

In [2]:
import sys
from pathlib import Path

import numpy as np

# The team's code lives in the repository root, one level above this folder.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "baseline.py").exists())
sys.path.insert(0, str(REPO))

from grid_world_env import GridWorld
from blackjack_env import Blackjack

from entropy_reg_spmd import ExactScorer

rng = np.random.default_rng(0)

### GridWorld

10 × 10 board, start in the top-left square (S = 0). Actions: up, down, left, right. The intended move
happens with probability 0.8; otherwise one of the other three directions is taken. Walls clip
movement. Each move costs 0.2 from an ordinary square, 1.0 from a hazard (H) and 0 from a goal (G).
From a goal or hazard the agent is sent back to the start with probability 0.8. Runs never
terminate, so the agent should reach a goal while avoiding hazards.

In [3]:
gridworld = GridWorld()
print(f"states: {gridworld.n_states}, actions: {gridworld.n_actions} {gridworld.action_names}, gamma: {gridworld.gamma}")

board = np.full((gridworld.rows, gridworld.cols), ".")
board.flat[list(gridworld.goals)] = "G"
board.flat[list(gridworld.hazards)] = "H"
board.flat[gridworld.start] = "S"
print("\n".join(" ".join(row) for row in board))

# One sampled step from the start for each action: (next state, cost, done).
states = gridworld.initial_states(4, rng)
next_states, costs, done = gridworld.step(states, np.arange(4), rng)
print("\nfrom the start:", [(name, int(s), float(c)) for name, s, c in zip(gridworld.action_names, next_states, costs)])

states: 100, actions: 4 ('up', 'down', 'left', 'right'), gamma: 0.9
S . . . . . . . . G
. . H . . . . H . .
. . . . H . . . . .
. . . . . . H . . .
. . . H . . . . . .
. . . . . . . . H .
. . H . . . . . . .
. . . . . H . . . .
. H . . . . . H . .
G . . . . . . . . G

from the start: [('up', 0, 0.2), ('down', 10, 0.2), ('left', 0, 0.2), ('right', 0, 0.2)]


### Blackjack

Simplified game from the project handout: cards 2–5 with equal probability, no aces. A state is
(player total 4–11, dealer's visible card 2–5), so 32 states. Actions: stop (0) or draw (1). Going
above 11 is a bust. When the player stops, the dealer draws until reaching 9 or more. Terminal cost:
win −1, draw 0, loss +1. Every episode starts from the fixed hand (total 8, dealer shows 3).

In [4]:
blackjack = Blackjack()
start = int(blackjack.encode(blackjack.evaluation_total, blackjack.evaluation_dealer))
print(f"states: {blackjack.n_states}, actions: {blackjack.n_actions} {blackjack.action_names}, gamma: {blackjack.gamma}")
print(f"start state: {start} = (total {blackjack.evaluation_total}, dealer {blackjack.evaluation_dealer})")

# Ten sampled "stop" moves from the start: the dealer plays out, cost -1 win / 0 draw / +1 loss.
states = blackjack.initial_states(10, rng)
_, costs, done = blackjack.step(states, np.zeros(10, dtype=int), rng)
print("stop from the start, 10 samples:", costs.astype(int).tolist(), "done:", bool(done.all()))

states: 32, actions: 2 ('stop', 'draw'), gamma: 0.9
start state: 17 = (total 8, dealer 3)
stop from the start, 10 samples: [1, 1, 1, 1, -1, 1, 1, 1, 1, 1] done: True


## Online validation: baseline SPMD vs Entropy-Regularized SPMD

Every method learns from **online sampling** with the team's policy evaluation baseline,
**Truncated On-Policy Monte Carlo with dynamic exploration time** (`baseline.tomc`). The baseline and
the enhancement differ only in the policy update. The runs live in `experiment.py` so they can use
several processes.

- **Estimator (`baseline.tomc`):** each iteration starts a fresh stream from the start state (blackjack
  deals a new hand after each finished one) and follows the current policy until every pair with
  π(a|s) ≥ (1−γ)/|A| has been tried and the discounted tail after the last first visit is below
  `bias` = 0.01. Q(s, a) is the return after the first visit to (s, a); rarer pairs get the
  pessimistic cost 1/(1−γ) = 10. The number of steps per iteration depends on the policy.
- **Baseline SPMD:** the team's KL-SPMD step (`baseline.spmd`), π ∝ π exp(−ηQ), which is
  `entropy_reg_spmd` with τ = 0.
- **Entropy-Regularized SPMD:** `entropy_reg_spmd` with τ > 0. Each sampled step's cost includes
  τ Σ_a π(a|s) log π(a|s), so the sampled Q is the regularized Q.
- **Budgets** (environment steps): GridWorld 3M (one estimate under the uniform policy takes 0.2M–1.4M
  steps), blackjack 120k (about 700 steps per estimate). If the budget runs out during an estimate,
  those steps give no update.
- **Protocol:**
  - Each method's step size is tuned on seeds 100–102 and tested on 20 fresh seeds (200–219).
  - After every update the policy is scored **exactly** from the model, with no sampling noise.
  - The score is the gap to the optimal cost from the start state (0 = optimal).

In [5]:
import os
from concurrent.futures import ProcessPoolExecutor

from entropy_reg_spmd.experiment import CURVE_POINTS, curve_job

TAUS = [0.001, 0.01, 0.03, 0.1]
METHODS = {"baseline SPMD": 0.0, **{f"PMD, tau={tau:g}": tau for tau in TAUS}}  # name -> tau
STEP_SIZES = [0.3, 1.0, 5.0, 20.0, 100.0]  # the same KL step for every method, so one grid
TUNING_SEEDS = [100, 101, 102]
TEST_SEEDS = list(range(200, 220))
BIAS = 0.01  # baseline.tomc: stop once the discounted tail after the last first visit is below this
WORKERS = max(1, (os.cpu_count() or 2) - 1)

# name: (environment class, budget in environment steps)
PROBLEMS = {
    "GridWorld": ("grid_world_env.GridWorld", 3_000_000),
    "Blackjack": ("blackjack_env.Blackjack", 120_000),
}


def curves_for(pool, environment, budget, settings, seeds):
    """{(tau, eta): array of gap curves, one per seed}, computed in parallel."""
    jobs = [(environment, tau, eta, BIAS, budget, seed) for tau, eta in settings for seed in seeds]
    results = iter(pool.map(curve_job, jobs, chunksize=1))
    return {setting: np.array([next(results) for _ in seeds]) for setting in settings}

In [ ]:
# Tune step sizes, then test on fresh seeds (about 2 hours on 7 processes, almost all GridWorld).
curves, chosen = {}, {}
with ProcessPoolExecutor(WORKERS) as pool:
    for problem, (environment, budget) in PROBLEMS.items():
        print(f"{problem}: budget {budget:,} steps")
        tuning = curves_for(pool, environment, budget, [(t, eta) for t in METHODS.values() for eta in STEP_SIZES], TUNING_SEEDS)
        best = {name: min(STEP_SIZES, key=lambda eta: tuning[tau, eta][:, -1].mean()) for name, tau in METHODS.items()}
        tested = curves_for(pool, environment, budget, [(tau, best[name]) for name, tau in METHODS.items()], TEST_SEEDS)
        for name, tau in METHODS.items():
            chosen[problem, name] = best[name]
            curves[problem, name] = tested[tau, best[name]]
            print(f"  {name:14s} step size {best[name]:<5g} final gap {curves[problem, name][:, -1].mean():.4f}")

GridWorld: budget 3,000,000 steps


### Results

Final gap over the 20 test seeds (mean ± standard error). The last columns compare each method with
baseline SPMD: z ≤ −2 counts as better, and z ≥ +2 with a difference of at least 0.001 counts as worse.

In [ ]:
print(f"{'problem':10s} {'method':15s} {'step':>5s} {'gap@25%':>8s} {'gap@50%':>8s} {'final gap':>16s} {'vs baseline':>12s} {'z':>6s}  verdict")
for problem in PROBLEMS:
    base = curves[problem, "baseline SPMD"][:, -1]
    for name in METHODS:
        runs = curves[problem, name]
        final = runs[:, -1]
        diff = final.mean() - base.mean()
        se_diff = np.sqrt(final.var(ddof=1) / len(final) + base.var(ddof=1) / len(base))
        z = diff / se_diff if se_diff > 0 else 0.0
        verdict = "" if name == "baseline SPMD" else (
            "better" if z <= -2 else "worse" if z >= 2 and diff >= 0.001 else "tie")
        final_text = f"{final.mean():.4f} ± {final.std(ddof=1) / np.sqrt(len(final)):.4f}"
        print(f"{problem:10s} {name:15s} {chosen[problem, name]:5g} {runs[:, 25].mean():8.4f} {runs[:, 50].mean():8.4f} "
              f"{final_text:>16s} {diff:+12.4f} {z:+6.1f}  {verdict}")
    print()

In [ ]:
import matplotlib.pyplot as plt

colors = {"baseline SPMD": "#eb6834", "PMD, tau=0": "#86b6ef", "PMD, tau=0.001": "#3987e5",
          "PMD, tau=0.01": "#256abf", "PMD, tau=0.03": "#184f95", "PMD, tau=0.1": "#0d366b"}
styles = {"PMD, tau=0": (0, (1, 1.5)), "PMD, tau=0.01": (0, (5, 2)), "PMD, tau=0.03": (0, (2, 1, 1, 1)),
          "PMD, tau=0.1": (0, (8, 2, 1, 2))}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), layout="constrained")
for ax, (problem, (environment, budget)) in zip(axes, PROBLEMS.items()):
    scale, unit = (1e6, "millions") if budget >= 1e6 else (1e3, "thousands")
    x = CURVE_POINTS * budget / scale
    late = 0
    for name in METHODS:
        runs = curves[problem, name]
        mean, se = runs.mean(0), runs.std(0, ddof=1) / np.sqrt(len(runs))
        ax.fill_between(x, mean - se, mean + se, color=colors[name], alpha=0.15, linewidth=0)
        ax.plot(x, mean, color=colors[name], linestyle=styles.get(name, "-"),
                linewidth=2.6 if name == "baseline SPMD" else 2, label=name)
        late = max(late, (mean + se)[CURVE_POINTS >= 0.1].max())
    start = curves[problem, "baseline SPMD"][:, 0].mean()
    ax.set_ylim(0, min(start, 1.15 * late))  # all methods start from the uniform policy's gap
    ax.set_xlim(0, x[-1])
    ax.set_title(f"{problem}, online sampling (all start at gap {start:.2f})", loc="left", fontsize=10)
    ax.set_xlabel(f"environment steps used ({unit})")
    ax.grid(color="#e1e0d9", linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("gap to optimal cost")
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=6, frameon=False)
fig.suptitle(f"Baseline SPMD vs Entropy-Regularized SPMD: mean of {len(TEST_SEEDS)} test seeds, band = ±1 SE (lower is better)")
plt.show()